In [81]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim 
from torch.utils.data import DataLoader,random_split,TensorDataset
from torchvision.utils import make_grid
from torchvision.transforms import ToTensor
from torchvision.datasets import ImageFolder
import torchvision.transforms as transforms
import torchvision.models as models

In [82]:
from tqdm.notebook import tqdm
import os

In [83]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [84]:
data =r"E:\datasets\Plant_leaves_detaset_CAM"
classes = os.listdir(data)

In [85]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(256, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),
    transforms.RandomRotation(25),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [86]:
dataset = ImageFolder(r"E:\datasets\Plant_leaves_detaset_CAM",transform=transform)
test_ds = ImageFolder(r"C:\Users\nayna\pd project\trainig\test", transform=val_transform)
val_ds = ImageFolder(r"C:\Users\nayna\pd project\trainig\val", transform=val_transform)

In [87]:
from collections import Counter

labels = [label for _, label in dataset.samples]
class_counts = Counter(labels)

for class_idx, count in class_counts.items():
    print(dataset.classes[class_idx], count)

Apple___Apple_scab 784
Apple___Black_rot 776
Apple___Cedar_apple_rust 500
Apple___healthy 672
Cauliflower_Black_Rot 708
Cauliflower_Healthy 806
Cherry_(including__sour)_Powdery_mildew 738
Cherry_(including_sour)___healthy 760
Corn_(maize)__Cercospora_leaf_spot_Gray_leaf_spot 690
Corn_(maize)__Common_rust 789
Corn_(maize)___Northern_Leaf_Blight 636
Corn_(maize)___healthy 501
Cotton_Aphids 280
Cotton_Bacterial_Blight 280
Cotton_Healthy 280
Cotton_Powdery_Mildew 280
Cotton_Target_spot 280
Grape___Black_rot 586
Grape___Esca_(Black_Measles) 573
Grape___Leaf_blight_(Isariopsis_Leaf_Spot) 415
Grape___healthy 498
Jackfruit_Algal_Leaf_Spot 280
Jackfruit_Black_Spot 280
Jackfruit_Healthy 280
Mango_Anthracnose 280
Mango_Bacterial_Canker 280
Mango_Cutting_Weevil 280
Mango_Die_Back 280
Mango_Gall_Midge 280
Mango_Healthy 280
Mango_Powdery_Mildew 280
Mango_Sooty_Mould 280
Peach___Bacterial_spot 508
Peach___healthy 568
Pepper__bell___Bacterial_spot 507
Pepper__bell___healthy 451
Potato___Early_blight 5

In [88]:
print(dataset.classes[:10])
print(val_ds.classes[:10])
print(dataset.classes == val_ds.classes)


['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Cauliflower_Black_Rot', 'Cauliflower_Healthy', 'Cherry_(including__sour)_Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)__Cercospora_leaf_spot_Gray_leaf_spot', 'Corn_(maize)__Common_rust']
['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Cauliflower_Black_Rot', 'Cauliflower_Healthy', 'Cherry_(including__sour)_Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)__Cercospora_leaf_spot_Gray_leaf_spot', 'Corn_(maize)__Common_rust']
True


In [89]:
print("Number of training images: ",len(dataset))
print("Number of testing images: ",len(test_ds))
print("number of val images:", len(val_ds))

Number of training images:  29468
Number of testing images:  5180
number of val images: 11900


In [90]:
num_classes = dataset.classes
print("Number of classes: ",len(num_classes))
print(num_classes)
print(val_ds.classes)
print("Number of classes: ",len(val_ds.classes))

Number of classes:  66
['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Cauliflower_Black_Rot', 'Cauliflower_Healthy', 'Cherry_(including__sour)_Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)__Cercospora_leaf_spot_Gray_leaf_spot', 'Corn_(maize)__Common_rust', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Cotton_Aphids', 'Cotton_Bacterial_Blight', 'Cotton_Healthy', 'Cotton_Powdery_Mildew', 'Cotton_Target_spot', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Jackfruit_Algal_Leaf_Spot', 'Jackfruit_Black_Spot', 'Jackfruit_Healthy', 'Mango_Anthracnose', 'Mango_Bacterial_Canker', 'Mango_Cutting_Weevil', 'Mango_Die_Back', 'Mango_Gall_Midge', 'Mango_Healthy', 'Mango_Powdery_Mildew', 'Mango_Sooty_Mould', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___

In [101]:
epochs = 25
random_seed = 42
torch.manual_seed(random_seed)

In [102]:
batch_size = 64

train_loader = DataLoader(dataset,batch_size=batch_size,num_workers=2,shuffle=True, pin_memory=True)
val_loader = DataLoader(val_ds,batch_size=batch_size,num_workers=2,shuffle=False, pin_memory=True)
test_loader = DataLoader(test_ds,batch_size=batch_size,num_workers=2,shuffle=True, pin_memory=True)

In [103]:
import torch
from collections import Counter

# train_dataset must already be created using ImageFolder
labels = [label for _, label in dataset.samples]

class_count = Counter(labels)

num_classes = len(dataset.classes)
total_samples = sum(class_count.values())

class_weights = torch.zeros(num_classes)

for i in range(num_classes):
    class_weights[i] = (total_samples / class_count[i])**0.5

# (optional but recommended)
class_weights = class_weights / class_weights.sum() * num_classes



In [104]:
# Accuracy helper
def accuracy(outputs, labels):
    _, preds = torch.max(outputs, dim=1)
    return torch.tensor(torch.sum(preds == labels).item() / len(preds))

# Base training/validation structure
criterion = nn.CrossEntropyLoss()
#criterion = nn.CrossEntropyLoss(
    #weight=class_weights.to(device),
    #label_smoothing=0.)

class ImageClassificationBase(nn.Module):

    def training_step(self, batch):
        images, labels = batch
        images, labels = images.to(device), labels.to(device)

        out = self(images)
        loss = criterion(out, labels)   # ✅ weighted loss
        acc = accuracy(out, labels)

        return {'train_loss': loss, 'train_acc': acc}

    def validation_step(self, batch):
        images, labels = batch
        images, labels = images.to(device), labels.to(device)

        out = self(images)
        loss = criterion(out, labels)   # ✅ same loss
        acc = accuracy(out, labels)

        return {'val_loss': loss, 'val_acc': acc}

    def validation_epoch_end(self, outputs):
        epoch_loss = torch.stack([x['val_loss'] for x in outputs]).mean()
        epoch_acc  = torch.stack([x['val_acc'] for x in outputs]).mean()

        return {
            'val_loss': epoch_loss.item(),
            'val_acc': epoch_acc.item()
        }

    def epoch_end(self, epoch, result):
        print(
            f"Epoch [{epoch+1}] "
            f"train_loss: {result['train_loss']:.4f}, "
            f"val_loss: {result['val_loss']:.4f}, "
            f"val_acc: {result['val_acc']:.4f}"
        )


In [105]:
import torch.nn as nn

class MyNN(ImageClassificationBase):
    def __init__(self, ip_features=3, num_classes = 66):
        super().__init__()
    
        self.features = nn.Sequential(
            nn.Conv2d(ip_features, 16, kernel_size=3, padding=1),  # (16, 256, 256)
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),                 # (16, 128, 128)

            nn.Conv2d(16, 32, kernel_size=3, padding=1),           # (32, 128, 128)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),                 # (32, 64, 64)

            nn.Conv2d(32, 64, kernel_size=3, padding=1),           # (64, 64, 64)
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),                 # (64, 32, 32)

            nn.Conv2d(64, 128, kernel_size=3, padding=1),          # (128, 32, 32)
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),                 # (128, 16, 16)

            nn.Conv2d(128, 256, kernel_size=3, padding=1),         # (256, 16, 16)
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)                  # (256, 8, 8)
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 8 * 8, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(512,num_classes)  # Make num_classes dynamic
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


In [106]:
# model
model = MyNN(ip_features=3, num_classes=num_classes).to(device)


optimizer = torch.optim.Adam(model.parameters(), lr=0.0007)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.3,
    patience=4,
)


In [107]:
from tqdm import tqdm

def fit(
    epochs,
    lr,
    model,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    patience=4,
    min_delta=0.002
):
    history = []

    best_val_acc = 0.0
    epochs_no_improve = 0

    for epoch in range(epochs):

        # 🔁 Training
        model.train()
        train_losses = []
        train_accs = []

        for batch in tqdm(train_loader):
            result = model.training_step(batch)

            loss = result['train_loss']
            acc = result['train_acc']

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            train_losses.append(loss.detach())
            train_accs.append(acc.detach())

        # 🔍 Validation
        val_result = evaluate(model, val_loader)

        val_result['train_loss'] = torch.stack(train_losses).mean().item()
        val_result['train_acc'] = torch.stack(train_accs).mean().item()

        # 📉 Scheduler step
        scheduler.step(val_result['val_acc'])

        # 💾 Early stopping + best model saving
        current_val_acc = val_result['val_acc']

        if current_val_acc > best_val_acc + min_delta:
            best_val_acc = current_val_acc
            epochs_no_improve = 0
            torch.save(model.state_dict(), "best_cnn.pth")
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"🛑 Early stopping triggered at epoch {epoch+1}")
            break

        # 📊 Logging
        model.epoch_end(epoch, val_result)
        history.append(val_result)

    return history


In [108]:
@torch.no_grad()
def evaluate(model, val_loader):
    model.eval()

    outputs = [model.validation_step(batch) for batch in val_loader]
    return model.validation_epoch_end(outputs)


In [ ]:
evaluate(model,val_loader)

In [109]:
history = fit(
    epochs=25,
    lr=0.0007,
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler
)



100%|████████████████████████████████████████████████████████████████████████████████| 461/461 [20:54<00:00,  2.72s/it]


Epoch [1] train_loss: 2.8046, val_loss: 2.2980, val_acc: 0.3590


100%|████████████████████████████████████████████████████████████████████████████████| 461/461 [20:22<00:00,  2.65s/it]


Epoch [2] train_loss: 1.8573, val_loss: 1.7849, val_acc: 0.4907


100%|████████████████████████████████████████████████████████████████████████████████| 461/461 [19:40<00:00,  2.56s/it]


...

Epoch [20] train_loss: 0.5202, val_loss: 0.8554, val_acc: 0.8281


100%|████████████████████████████████████████████████████████████████████████████████| 461/461 [20:25<00:00,  2.66s/it]


Epoch [21] train_loss: 0.5080, val_loss: 0.8865, val_acc: 0.8443


100%|████████████████████████████████████████████████████████████████████████████████| 461/461 [19:24<00:00,  2.53s/it]


🛑 Early stopping triggered at epoch 22


In [99]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels[:20])
print(labels.min(), labels.max())


torch.Size([64, 3, 256, 256])
tensor([58, 22, 59,  1, 47,  8,  1,  5,  6, 55, 38, 45, 63, 53,  3, 59,  0, 33,
         2, 55])
tensor(0) tensor(65)
